# 05c. Modelo Híbrido

Modelo principal recomendado: mezcla señales colaborativas con similitud de contenido derivada de géneros, tags y metadatos enriquecidos.


In [ ]:
from pathlib import Path
import sys
import warnings

warnings.filterwarnings('ignore')

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from IPython.display import display
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (10, 5)

import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

from src.utils.recsys_utils import build_item_knn_model, ensure_dir, make_binary_ground_truth, precision_recall_ndcg_at_k

processed_dir = ROOT / 'data' / 'processed'
tables_dir = ensure_dir(ROOT / 'reports' / 'tables')
models_dir = ensure_dir(ROOT / 'models')

train = pd.read_parquet(processed_dir / 'interactions_train.parquet')
test = pd.read_parquet(processed_dir / 'interactions_test.parquet')
item_features = pd.read_parquet(processed_dir / 'item_features.parquet')


## Definir subconjunto comparable con CF

Igual que en el notebook de CF, trabajamos con usuarios e ítems densos para prototipar y comparar modelos bajo el mismo régimen.


In [ ]:
active_users = train.groupby('userId').size().loc[lambda s: s >= 50].index
active_items = train.groupby('movieId').size().loc[lambda s: s >= 100].index
train_dense = train[train['userId'].isin(active_users) & train['movieId'].isin(active_items)].copy()
test_dense = test[test['userId'].isin(active_users) & test['movieId'].isin(active_items)].copy()

reuser = pd.DataFrame({'userId': sorted(train_dense['userId'].unique())})
reuser['user_idx'] = np.arange(len(reuser))
reitem = pd.DataFrame({'movieId': sorted(train_dense['movieId'].unique())})
reitem['item_idx'] = np.arange(len(reitem))

train_dense = train_dense.merge(reuser, on='userId', how='inner').merge(reitem, on='movieId', how='inner')
test_dense = test_dense.merge(reuser, on='userId', how='inner').merge(reitem, on='movieId', how='inner')
feature_subset = item_features[item_features['movieId'].isin(reitem['movieId'])].merge(reitem, on='movieId', how='inner').sort_values('item_idx')


## Score colaborativo + score de contenido

El score híbrido combina vecinos colaborativos con similitud coseno sobre features de contenido.


In [ ]:
cf_model, item_matrix = build_item_knn_model(
    train_dense,
    n_users=train_dense['user_idx'].nunique(),
    n_items=train_dense['item_idx'].nunique(),
    n_neighbors=30,
)

content_feature_cols = [
    col for col in feature_subset.columns
    if col not in {'movieId', 'item_idx', 'title', 'clean_title'} and pd.api.types.is_numeric_dtype(feature_subset[col])
]
content_matrix = feature_subset[content_feature_cols].fillna(0.0).to_numpy(dtype=float)
content_similarity = cosine_similarity(content_matrix)

positive_train = train_dense[train_dense['rating'] >= 4.0].copy()
seen_items = positive_train.groupby('user_idx')['item_idx'].apply(set).to_dict()

cf_neighbors = {}
for item_idx in train_dense['item_idx'].unique():
    distances, indices = cf_model.kneighbors(item_matrix[item_idx], n_neighbors=16)
    cf_neighbors[int(item_idx)] = [
        (int(idx), float(1 - dist))
        for idx, dist in zip(indices[0], distances[0])
        if int(idx) != int(item_idx)
    ]

recommendation_rows = []
alpha = 0.7
for user_idx, user_items in positive_train.groupby('user_idx')['item_idx']:
    scores = {}
    for item_idx in user_items.tolist():
        for neighbor_idx, cf_score in cf_neighbors[int(item_idx)]:
            if neighbor_idx in seen_items.get(int(user_idx), set()):
                continue
            content_score = float(content_similarity[int(item_idx), int(neighbor_idx)])
            hybrid_score = alpha * cf_score + (1 - alpha) * content_score
            scores[neighbor_idx] = scores.get(neighbor_idx, 0.0) + hybrid_score
    top_items = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:10]
    for rank, (item_idx, score) in enumerate(top_items, start=1):
        recommendation_rows.append({'user_idx': int(user_idx), 'item_idx': int(item_idx), 'score': float(score), 'rank': rank})

recommendations_hybrid = pd.DataFrame(recommendation_rows)
display(recommendations_hybrid.head())


## Evaluación del híbrido

El mismo esquema de evaluación permite comparar directamente con CF.


In [ ]:
ground_truth = make_binary_ground_truth(test_dense, threshold=4.0)
metrics = precision_recall_ndcg_at_k(recommendations_hybrid, ground_truth, k=10)
metrics_hybrid = pd.DataFrame([{'model': 'hybrid_cf_content', **metrics}])
display(metrics_hybrid)

recommendations_hybrid.to_parquet(models_dir / 'hybrid_recommendations.parquet', index=False)
metrics_hybrid.to_csv(tables_dir / 'hybrid_metrics.csv', index=False)
